# NB_bishop_ch14_figures
## Key Figures for Bishop Chapter 14 -- Sampling Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_14/NB_bishop_ch14_figures.ipynb)

Generate publication-quality figures for rejection sampling, importance sampling, MCMC, Gibbs sampling, and Hamiltonian Monte Carlo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
from scipy import stats

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 14.1 -- Rejection Sampling

Proposal distribution $kq(x)$ envelopes the target $p(x)$. Samples drawn from $q(x)$ are accepted if a uniform draw falls below $p(x)/kq(x)$, otherwise rejected.

In [ ]:
# Target: mixture of two Gaussians
x = np.linspace(-5, 7, 500)
p_x = 0.4 * stats.norm.pdf(x, 0, 1) + 0.6 * stats.norm.pdf(x, 3, 0.8)

# Proposal: broad Gaussian
q_x = stats.norm.pdf(x, 1.5, 2.2)
k = 1.15 * (p_x / q_x).max()  # scale factor
kq_x = k * q_x

fig, ax = plt.subplots(figsize=(10, 5))

# Fill the region between p(x) and kq(x) -- rejection region
ax.fill_between(x, p_x, kq_x, alpha=0.15, color=COLORS['red'], label='Rejection region')
ax.fill_between(x, 0, p_x, alpha=0.15, color=COLORS['green'], label='Acceptance region')

ax.plot(x, p_x, color=COLORS['blue'], linewidth=2.5, label='Target $p(x)$')
ax.plot(x, kq_x, color=COLORS['red'], linewidth=2.5, linestyle='--', label='Envelope $kq(x)$')

# Generate samples and classify
np.random.seed(42)
n_samples = 200
x_prop = np.random.normal(1.5, 2.2, n_samples)
u = np.random.uniform(0, 1, n_samples)

p_at_x = 0.4 * stats.norm.pdf(x_prop, 0, 1) + 0.6 * stats.norm.pdf(x_prop, 3, 0.8)
kq_at_x = k * stats.norm.pdf(x_prop, 1.5, 2.2)
accept = u < (p_at_x / kq_at_x)

# Plot accepted/rejected points
y_pts = u * kq_at_x  # vertical position of the uniform draw
ax.scatter(x_prop[accept], y_pts[accept], color=COLORS['green'], s=18,
           alpha=0.7, zorder=5, label=f'Accepted ({accept.sum()})')
ax.scatter(x_prop[~accept], y_pts[~accept], color=COLORS['red'], s=18,
           alpha=0.5, zorder=5, marker='x', label=f'Rejected ({(~accept).sum()})')

ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Rejection Sampling', fontweight='bold', fontsize=14)
ax.legend(fontsize=9)
ax.set_xlim(-5, 7)
ax.set_ylim(0, kq_x.max() * 1.1)

fig.tight_layout()
save_fig(fig, 'fig14_1_rejection_sampling')
plt.show()

## Figure 14.2 -- Importance Sampling

Target $p(x)$ versus proposal $q(x)$, with importance weights $w(x) = p(x)/q(x)$ visualized as bar heights at sample locations.

In [ ]:
# Target: mixture of Gaussians; Proposal: single Gaussian
x = np.linspace(-4, 8, 500)
p_x = 0.5 * stats.norm.pdf(x, 1, 0.8) + 0.5 * stats.norm.pdf(x, 4, 1.0)
q_x = stats.norm.pdf(x, 2.5, 2.0)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

# Top panel: distributions
ax1 = axes[0]
ax1.plot(x, p_x, color=COLORS['blue'], linewidth=2.5, label='Target $p(x)$')
ax1.plot(x, q_x, color=COLORS['red'], linewidth=2.5, linestyle='--', label='Proposal $q(x)$')
ax1.fill_between(x, 0, p_x, alpha=0.1, color=COLORS['blue'])
ax1.fill_between(x, 0, q_x, alpha=0.1, color=COLORS['red'])

# Draw samples from q and show them
np.random.seed(14)
n_is = 30
x_samples = np.random.normal(2.5, 2.0, n_is)
x_samples = x_samples[(x_samples > -3.5) & (x_samples < 7.5)]
p_at_s = 0.5 * stats.norm.pdf(x_samples, 1, 0.8) + 0.5 * stats.norm.pdf(x_samples, 4, 1.0)
q_at_s = stats.norm.pdf(x_samples, 2.5, 2.0)

ax1.scatter(x_samples, q_at_s, color=COLORS['red'], s=30, zorder=5, edgecolors='black', linewidths=0.5)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Importance Sampling', fontweight='bold', fontsize=14)
ax1.legend(fontsize=10)

# Bottom panel: importance weights
ax2 = axes[1]
weights = p_at_s / q_at_s
# Normalize weights for display
w_norm = weights / weights.max()

bars = ax2.bar(x_samples, weights, width=0.15, color=COLORS['amber'], alpha=0.8,
               edgecolor='black', linewidth=0.8)

ax2.set_xlabel('$x$', fontsize=12)
ax2.set_ylabel('Weight $w(x)$', fontsize=12)
ax2.set_title('Importance Weights $w(x) = p(x) / q(x)$', fontweight='bold', fontsize=12)
ax2.axhline(y=1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax2.set_xlim(-4, 8)

fig.tight_layout()
save_fig(fig, 'fig14_2_importance_sampling')
plt.show()

## Figure 14.3 -- MCMC Trajectory

Random walk Metropolis--Hastings on a 2D target distribution. The trace of accepted samples is overlaid on contour lines of the target density.

In [ ]:
# 2D target: correlated bivariate Gaussian
mean = np.array([2.0, 3.0])
cov = np.array([[1.0, 0.8], [0.8, 1.0]])
cov_inv = np.linalg.inv(cov)
cov_det = np.linalg.det(cov)

def target_pdf(x1, x2):
    d = np.array([x1 - mean[0], x2 - mean[1]])
    return np.exp(-0.5 * d @ cov_inv @ d) / (2 * np.pi * np.sqrt(cov_det))

# Random walk Metropolis-Hastings
np.random.seed(42)
n_steps = 500
proposal_std = 0.5
chain = np.zeros((n_steps, 2))
chain[0] = [0.0, 0.0]  # start away from mode
accepted = [True]

for t in range(1, n_steps):
    proposal = chain[t-1] + np.random.normal(0, proposal_std, 2)
    p_current = target_pdf(chain[t-1, 0], chain[t-1, 1])
    p_proposal = target_pdf(proposal[0], proposal[1])
    alpha = min(1.0, p_proposal / max(p_current, 1e-300))
    if np.random.uniform() < alpha:
        chain[t] = proposal
        accepted.append(True)
    else:
        chain[t] = chain[t-1]
        accepted.append(False)

accepted = np.array(accepted)

# Create contour grid
x1_grid = np.linspace(-1, 5, 200)
x2_grid = np.linspace(0, 6, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Z = np.zeros_like(X1)
for i in range(X1.shape[0]):
    for j in range(X1.shape[1]):
        Z[i, j] = target_pdf(X1[i, j], X2[i, j])

fig, ax = plt.subplots(figsize=(8, 7))

# Contours
ax.contourf(X1, X2, Z, levels=15, cmap='Blues', alpha=0.4)
ax.contour(X1, X2, Z, levels=10, colors=COLORS['blue'], alpha=0.5, linewidths=0.8)

# Chain trajectory
burn_in = 50
ax.plot(chain[:burn_in, 0], chain[:burn_in, 1], color=COLORS['red'],
        alpha=0.5, linewidth=0.8, label=f'Burn-in ({burn_in} steps)')
ax.plot(chain[burn_in:, 0], chain[burn_in:, 1], color=COLORS['blue'],
        alpha=0.3, linewidth=0.5)

# Mark accepted vs rejected after burn-in
ax.scatter(chain[burn_in:, 0][accepted[burn_in:]], chain[burn_in:, 1][accepted[burn_in:]],
           color=COLORS['green'], s=8, alpha=0.6, zorder=4, label='Accepted')

# Start and end markers
ax.scatter(*chain[0], color=COLORS['red'], s=120, marker='*', zorder=6,
           edgecolors='black', linewidths=1, label='Start')
ax.scatter(*chain[-1], color=COLORS['green'], s=120, marker='D', zorder=6,
           edgecolors='black', linewidths=1, label='End')

# True mean
ax.scatter(*mean, color=COLORS['amber'], s=150, marker='+', zorder=6,
           linewidths=3, label='True mean')

ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_title('Random Walk Metropolis--Hastings', fontweight='bold', fontsize=14)
ax.legend(fontsize=9)

acc_rate = accepted[burn_in:].mean() * 100
ax.text(0.02, 0.02, f'Acceptance rate: {acc_rate:.0f}%',
        transform=ax.transAxes, fontsize=10, color=COLORS['blue'],
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

fig.tight_layout()
save_fig(fig, 'fig14_3_mcmc_trajectory')
plt.show()

## Figure 14.5 -- Gibbs Sampling

Gibbs sampling on a correlated 2D Gaussian: the sampler alternates between updating $x_1$ (horizontal moves) and $x_2$ (vertical moves), producing a characteristic zig-zag trajectory.

In [ ]:
# Gibbs sampling on correlated 2D Gaussian
mu = np.array([0.0, 0.0])
rho = 0.9
sigma1, sigma2 = 1.0, 1.0

np.random.seed(7)
n_gibbs = 40
gibbs_chain = np.zeros((2 * n_gibbs + 1, 2))
gibbs_chain[0] = [-2.5, -2.5]  # start far from mode

for t in range(n_gibbs):
    # Update x1 | x2
    x2_cur = gibbs_chain[2 * t, 1]
    cond_mean_1 = mu[0] + rho * sigma1 / sigma2 * (x2_cur - mu[1])
    cond_std_1 = sigma1 * np.sqrt(1 - rho**2)
    new_x1 = np.random.normal(cond_mean_1, cond_std_1)
    gibbs_chain[2 * t + 1] = [new_x1, x2_cur]

    # Update x2 | x1
    x1_cur = new_x1
    cond_mean_2 = mu[1] + rho * sigma2 / sigma1 * (x1_cur - mu[0])
    cond_std_2 = sigma2 * np.sqrt(1 - rho**2)
    new_x2 = np.random.normal(cond_mean_2, cond_std_2)
    gibbs_chain[2 * t + 2] = [x1_cur, new_x2]

# Contour of target
cov_gibbs = np.array([[sigma1**2, rho*sigma1*sigma2],
                       [rho*sigma1*sigma2, sigma2**2]])
x1g = np.linspace(-3.5, 3.5, 200)
x2g = np.linspace(-3.5, 3.5, 200)
X1g, X2g = np.meshgrid(x1g, x2g)
cov_inv_g = np.linalg.inv(cov_gibbs)
Zg = np.zeros_like(X1g)
for i in range(X1g.shape[0]):
    for j in range(X1g.shape[1]):
        d = np.array([X1g[i, j] - mu[0], X2g[i, j] - mu[1]])
        Zg[i, j] = np.exp(-0.5 * d @ cov_inv_g @ d)

fig, ax = plt.subplots(figsize=(8, 7))

ax.contourf(X1g, X2g, Zg, levels=12, cmap='Blues', alpha=0.35)
ax.contour(X1g, X2g, Zg, levels=8, colors=COLORS['blue'], alpha=0.4, linewidths=0.8)

# Draw zig-zag trajectory
for t in range(len(gibbs_chain) - 1):
    x_from, y_from = gibbs_chain[t]
    x_to, y_to = gibbs_chain[t + 1]
    is_horizontal = (t % 2 == 0)  # even steps update x1 (horizontal)
    color = COLORS['red'] if is_horizontal else COLORS['green']
    ax.plot([x_from, x_to], [y_from, y_to], color=color, linewidth=1.2, alpha=0.7)

# Draw sample points (at full Gibbs steps only)
full_steps = gibbs_chain[0::2]
ax.scatter(full_steps[:, 0], full_steps[:, 1], color=COLORS['amber'],
           s=25, zorder=5, edgecolors='black', linewidths=0.5)

# Start marker
ax.scatter(*gibbs_chain[0], color=COLORS['red'], s=120, marker='*',
           zorder=6, edgecolors='black', linewidths=1, label='Start')

# Legend entries for directions
ax.plot([], [], color=COLORS['red'], linewidth=2, label='Sample $x_1 | x_2$ (horizontal)')
ax.plot([], [], color=COLORS['green'], linewidth=2, label='Sample $x_2 | x_1$ (vertical)')
ax.scatter([], [], color=COLORS['amber'], s=30, edgecolors='black',
           linewidths=0.5, label='Full Gibbs samples')

ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_title(f'Gibbs Sampling ($\\rho = {rho}$)', fontweight='bold', fontsize=14)
ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, 'fig14_5_gibbs_sampling')
plt.show()

## Figure 14.7 -- Hamiltonian Monte Carlo

HMC augments the state with momentum variables and simulates Hamiltonian dynamics via leapfrog integration, producing long-range proposals with high acceptance probability.

In [ ]:
# HMC on a 2D correlated Gaussian
mu_hmc = np.array([0.0, 0.0])
cov_hmc = np.array([[1.0, 0.95], [0.95, 1.0]])
cov_inv_hmc = np.linalg.inv(cov_hmc)

def U(q):
    """Potential energy = -log p(q)"""
    d = q - mu_hmc
    return 0.5 * d @ cov_inv_hmc @ d

def grad_U(q):
    """Gradient of potential energy"""
    return cov_inv_hmc @ (q - mu_hmc)

def leapfrog(q0, p0, step_size, n_leapfrog):
    """Leapfrog integrator; returns trajectory for visualization."""
    trajectory = [q0.copy()]
    q = q0.copy()
    p = p0.copy()
    p -= 0.5 * step_size * grad_U(q)
    for i in range(n_leapfrog):
        q += step_size * p
        trajectory.append(q.copy())
        if i < n_leapfrog - 1:
            p -= step_size * grad_U(q)
    p -= 0.5 * step_size * grad_U(q)
    return q, p, np.array(trajectory)

# Run HMC
np.random.seed(42)
n_hmc = 15
step_size = 0.18
n_leapfrog = 25
hmc_samples = [np.array([-2.0, -2.0])]
trajectories = []

for _ in range(n_hmc):
    q_current = hmc_samples[-1].copy()
    p_current = np.random.randn(2)
    q_prop, p_prop, traj = leapfrog(q_current, p_current, step_size, n_leapfrog)
    trajectories.append(traj)

    # Metropolis acceptance
    H_current = U(q_current) + 0.5 * np.sum(p_current**2)
    H_prop = U(q_prop) + 0.5 * np.sum(p_prop**2)
    if np.random.uniform() < np.exp(min(0, -(H_prop - H_current))):
        hmc_samples.append(q_prop)
    else:
        hmc_samples.append(q_current)

hmc_samples = np.array(hmc_samples)

# Contour
x1h = np.linspace(-3.5, 3.5, 200)
x2h = np.linspace(-3.5, 3.5, 200)
X1h, X2h = np.meshgrid(x1h, x2h)
Zh = np.zeros_like(X1h)
for i in range(X1h.shape[0]):
    for j in range(X1h.shape[1]):
        d = np.array([X1h[i, j], X2h[i, j]])
        Zh[i, j] = np.exp(-U(d))

fig, ax = plt.subplots(figsize=(8, 7))

ax.contourf(X1h, X2h, Zh, levels=12, cmap='Blues', alpha=0.35)
ax.contour(X1h, X2h, Zh, levels=8, colors=COLORS['blue'], alpha=0.4, linewidths=0.8)

# Draw leapfrog trajectories
traj_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, n_hmc))
for idx, traj in enumerate(trajectories):
    ax.plot(traj[:, 0], traj[:, 1], color=traj_colors[idx],
            linewidth=1.0, alpha=0.6, zorder=3)
    # Arrow at end of trajectory
    if len(traj) > 1:
        ax.annotate('', xy=(traj[-1, 0], traj[-1, 1]),
                    xytext=(traj[-2, 0], traj[-2, 1]),
                    arrowprops=dict(arrowstyle='->', color=traj_colors[idx],
                                    lw=1.5, alpha=0.7))

# Draw accepted samples
ax.scatter(hmc_samples[:, 0], hmc_samples[:, 1], color=COLORS['amber'],
           s=50, zorder=5, edgecolors='black', linewidths=1, label='HMC samples')

# Connect samples
ax.plot(hmc_samples[:, 0], hmc_samples[:, 1], color=COLORS['amber'],
        linewidth=0.8, alpha=0.4, linestyle='--')

# Start
ax.scatter(*hmc_samples[0], color=COLORS['red'], s=150, marker='*',
           zorder=6, edgecolors='black', linewidths=1, label='Start')

# Legend for trajectories
ax.plot([], [], color=COLORS['green'], linewidth=2, alpha=0.7, label='Leapfrog trajectories')

ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_title('Hamiltonian Monte Carlo', fontweight='bold', fontsize=14)
ax.legend(fontsize=9)

ax.text(0.02, 0.02, f'Leapfrog: L={n_leapfrog}, $\\epsilon$={step_size}',
        transform=ax.transAxes, fontsize=10, color=COLORS['blue'],
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

fig.tight_layout()
save_fig(fig, 'fig14_7_hmc')
plt.show()

## Summary

- **fig14_1**: Rejection sampling with proposal envelope, accepted/rejected points
- **fig14_2**: Importance sampling with target vs proposal and weight bars
- **fig14_3**: Random walk Metropolis--Hastings trajectory on 2D Gaussian contours
- **fig14_5**: Gibbs sampling zig-zag trajectory on correlated 2D Gaussian
- **fig14_7**: Hamiltonian Monte Carlo with leapfrog trajectories on energy landscape